# 📦 Pipeline S3: Raw (CSV) & Bronze (Parquet)

Este notebook realiza:
1. **Conexão** com o AWS S3 utilizando as credenciais configuradas no `.env` e a função `iniciar_cessao_aws` de `src.utils`.
2. **Camada Raw**: Upload dos dados brutos em `.csv` do Kaggle (`olistbr/brazilian-ecommerce`) para `s3://<bucket>/raw/`.
3. **Camada Bronze**: Conversão dos datasets para formato colunar `.parquet` utilizando a função `salvar_parquet_s3` de `src.utils` e salvamento em `s3://<bucket>/bronze/`.
4. **Validação**: Verificação dos objetos gravados no S3 em ambas as camadas.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Localiza a raiz do projeto e adiciona ao sys.path para importar src.utils
notebook_dir = Path.cwd().resolve()
project_root = notebook_dir.parents[1] if notebook_dir.name == 'notebooks' else notebook_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import iniciar_cessao_aws, salvar_parquet_s3

# Carrega variáveis do arquivo .env
load_dotenv(find_dotenv())

BUCKET_NAME = os.getenv('BUCKET_NAME')
AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')

print(f'Projeto Root: {project_root}')
print(f'Bucket configurado: {BUCKET_NAME}')
print(f'Região: {AWS_REGION}')

Projeto Root: C:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_04
Bucket configurado: postech-challenge-datascience-004
Região: us-east-1


In [2]:
# Inicializa sessão e cliente S3 usando iniciar_cessao_aws() de src.utils
session = iniciar_cessao_aws()
s3_client = session.client('s3')

# Valida acesso ao bucket
buckets = [b['Name'] for b in s3_client.list_buckets().get('Buckets', [])]
print(f"Bucket '{BUCKET_NAME}' disponível na conta AWS? {BUCKET_NAME in buckets}")

Bucket 'postech-challenge-datascience-004' disponível na conta AWS? True


--- 
##  Camada Raw (Upload dos CSVs Brutos)

In [3]:
import kagglehub

# Localiza arquivos CSV locais (fase_4/data/raw) ou faz download via kagglehub
data_raw_path = Path('../data/raw').resolve()

if data_raw_path.exists() and any(data_raw_path.glob('*.csv')):
    source_dir = data_raw_path
    print(f'Utilizando arquivos locais de: {source_dir}')
else:
    print('Baixando base do Kaggle via kagglehub...')
    source_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
    print(f'Arquivos baixados em: {source_dir}')

csv_files = sorted(list(source_dir.glob('*.csv')))
print(f'\nTotal de arquivos CSV encontrados: {len(csv_files)}')
for f in csv_files:
    print(f'  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)')

Utilizando arquivos locais de: C:\Users\deth_\Carmel Capital\TECNOLOGIA - Geral\LUCAS\Estudos\FIAP\POSTECH_AI_SCIENTIST\tech_challenge_04\fase_4\data\raw

Total de arquivos CSV encontrados: 9
  - olist_customers_dataset.csv (8.62 MB)
  - olist_geolocation_dataset.csv (58.44 MB)
  - olist_order_items_dataset.csv (14.72 MB)
  - olist_order_payments_dataset.csv (5.51 MB)
  - olist_order_reviews_dataset.csv (13.78 MB)
  - olist_orders_dataset.csv (16.84 MB)
  - olist_products_dataset.csv (2.27 MB)
  - olist_sellers_dataset.csv (0.17 MB)
  - product_category_name_translation.csv (0.00 MB)


c:\Users\deth_\anaconda3\envs\fase_4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Upload dos arquivos CSV para s3://<bucket>/raw/
prefix_raw = 'raw'
print(f'Iniciando upload para s3://{BUCKET_NAME}/{prefix_raw}/\n')

for file_path in csv_files:
    s3_key = f'{prefix_raw}/{file_path.name}'
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f'Enviando {file_path.name} ({file_size_mb:.2f} MB) -> {s3_key}...', end=' ')
    s3_client.upload_file(
        Filename=str(file_path),
        Bucket=BUCKET_NAME,
        Key=s3_key
    )
    print('Sucesso!')

print('\n Upload dos CSVs brutos finalizado com sucesso!')

Iniciando upload para s3://postech-challenge-datascience-004/raw/

Enviando olist_customers_dataset.csv (8.62 MB) -> raw/olist_customers_dataset.csv... Sucesso!
Enviando olist_geolocation_dataset.csv (58.44 MB) -> raw/olist_geolocation_dataset.csv... Sucesso!
Enviando olist_order_items_dataset.csv (14.72 MB) -> raw/olist_order_items_dataset.csv... Sucesso!
Enviando olist_order_payments_dataset.csv (5.51 MB) -> raw/olist_order_payments_dataset.csv... Sucesso!
Enviando olist_order_reviews_dataset.csv (13.78 MB) -> raw/olist_order_reviews_dataset.csv... Sucesso!
Enviando olist_orders_dataset.csv (16.84 MB) -> raw/olist_orders_dataset.csv... Sucesso!
Enviando olist_products_dataset.csv (2.27 MB) -> raw/olist_products_dataset.csv... Sucesso!
Enviando olist_sellers_dataset.csv (0.17 MB) -> raw/olist_sellers_dataset.csv... Sucesso!
Enviando product_category_name_translation.csv (0.00 MB) -> raw/product_category_name_translation.csv... Sucesso!

 Upload dos CSVs brutos finalizado com sucesso!


--- 
## Camada Bronze (Conversão para Parquet via `salvar_parquet_s3`)

In [5]:
# Itera sobre os CSVs, carrega em DataFrame e salva como Parquet na camada bronze via utils.py
prefix_bronze = 'bronze'
print(f'Iniciando conversão e salvamento na camada Bronze: s3://{BUCKET_NAME}/{prefix_bronze}/\n')

for file_path in csv_files:
    table_name = file_path.stem  # ex: olist_customers_dataset
    chave_parquet = f'{prefix_bronze}/{table_name}.parquet'
    
    print(f'Processando {file_path.name}...', end=' ')
    df = pd.read_csv(file_path, low_memory=False)
    
    # Salva diretamente no S3 em formato Parquet higienizado usando src.utils
    salvar_parquet_s3(
        df=df,
        s3_client=s3_client,
        bucket=BUCKET_NAME,
        chave=chave_parquet
    )

print('\n  Todos os arquivos foram convertidos e salvos na camada Bronze com sucesso!')

Iniciando conversão e salvamento na camada Bronze: s3://postech-challenge-datascience-004/bronze/

Processando olist_customers_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_customers_dataset.parquet
Processando olist_geolocation_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_geolocation_dataset.parquet
Processando olist_order_items_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_order_items_dataset.parquet
Processando olist_order_payments_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_order_payments_dataset.parquet
Processando olist_order_reviews_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_order_reviews_dataset.parquet
Processando olist_orders_dataset.csv... Salvo no S3: s3://postech-challenge-datascience-004/bronze/olist_orders_dataset.parquet
Processando olist_products_dataset.csv... Salvo no S3: s3://postech-challenge

--- 
## Validação dos Objetos no S3 (Raw & Bronze)

In [6]:
# Listagem dos arquivos nas camadas raw e bronze
for camada in ['raw', 'bronze']:
    print(f'\n  Camada: {camada.upper()} (s3://{BUCKET_NAME}/{camada}/)')
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f'{camada}/')
    contents = response.get('Contents', [])
    if not contents:
        print('  (vazio)')
    for obj in contents:
        size_mb = obj['Size'] / (1024 * 1024)
        print(f"  - {obj['Key']} ({size_mb:.2f} MB)")


  Camada: RAW (s3://postech-challenge-datascience-004/raw/)
  - raw/olist_customers_dataset.csv (8.62 MB)
  - raw/olist_geolocation_dataset.csv (58.44 MB)
  - raw/olist_order_items_dataset.csv (14.72 MB)
  - raw/olist_order_payments_dataset.csv (5.51 MB)
  - raw/olist_order_reviews_dataset.csv (13.78 MB)
  - raw/olist_orders_dataset.csv (16.84 MB)
  - raw/olist_products_dataset.csv (2.27 MB)
  - raw/olist_sellers_dataset.csv (0.17 MB)
  - raw/product_category_name_translation.csv (0.00 MB)

  Camada: BRONZE (s3://postech-challenge-datascience-004/bronze/)
  - bronze/olist_customers_dataset.parquet (6.68 MB)
  - bronze/olist_geolocation_dataset.parquet (16.24 MB)
  - bronze/olist_order_items_dataset.parquet (6.24 MB)
  - bronze/olist_order_payments_dataset.parquet (3.66 MB)
  - bronze/olist_order_reviews_dataset.parquet (9.16 MB)
  - bronze/olist_orders_dataset.parquet (10.28 MB)
  - bronze/olist_products_dataset.parquet (1.34 MB)
  - bronze/olist_sellers_dataset.parquet (0.13 MB)
  - 